# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following : 

- Tracking agent behaviour with logging, analytics and debugging.
- Transforming primpts, tool selection, and output formatting. 
- Adding retries, fallbacks and early termination logic
- Applying rate limits, guardrails and PII detection

# Built-in Middleware 

- Summarization
- Human-in-the-loop
etc

In [6]:
import os
from dotenv import load_dotenv 
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters. 

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization
agent = create_agent(
    model="gpt-5.5",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model = ("gpt-4o-mini"),
        trigger = ("messages",10), #Message length
        keep = ("messages", 4)
        )]
)

In [14]:
# Run with the thread id 

config = {"configurable":{"thread_id":"test-1"}}

In [20]:
# Alterative test data 

questions = [
    "what is 2+2",
    "what is 5/10",
    "what is 7*7",
    "what is 4+4",
    "what is 9x2",
    "what is 2-9"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages : {response}")
    print(f"Messages : {len(response['messages'])}")

Messages : {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is requesting the answers to basic arithmetic problems.\n\n## SUMMARY\nThe user asked for the results of several mathematical operations, including addition, division, and multiplication. The responses provided were correct: \n- 2 + 2 = 4 \n- 5 / 10 = 0.5 (or 1/2)\n- 7 × 7 = 49\n\nThe user repeated the request for 2 + 2 again.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nRespond to the user's repeated request for the answer to 2 + 2.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='c985c027-6b32-4e89-8e8a-fb7b04f34394'), AIMessage(content='2 + 2 = 4', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 89, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_d

In [43]:
# now based on token size 

from langchain_core.tools import tool
@tool
def search_hotels(Cityv: str) -> str:
    """search hotels - return long response to use more tokens"""
    return f"""Hotels in {city}
    1. Grand Hotel - 5 star, ₹1000O/night, spa, pool, gym
    2.  City Inn - 4 star, ₹700/night, business center
    3.. Budget Stay - 3 star, ₹200/night, free wifi
    """

agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model = ("gpt-4o-mini"),
        trigger = ("tokens",300), #now token length 
        keep = ("tokens", 200) # recent 200 tokens
        )]
            
)

In [ ]:
config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars = 1 token

# run test
cities = ["hyderabad", "seoul", "gangnam", "mumbai", "pune", "degu", "busan", "kolkota"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config=config,
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, len(response['messages'])={len(response['messages'])} messages")
    print(response["messages"])


hyderabad: ~400 tokens, len(response['messages'])=7 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in Seoul, specifically in the Gangnam area.\n\n## SUMMARY\nThe user initially requested hotels in Hyderabad, for which a list was provided with details including star ratings, prices per night, and available amenities. The hotels listed were:\n1. Grand Hotel - 5 stars, ₹10,000/night, spa, pool, gym\n2. City Inn - 4 stars, ₹700/night, business center\n3. Budget Stay - 3 stars, ₹200/night, free WiFi  \nAfter discussing Hyderabad, the user shifted focus to seeking hotels in Seoul. A subsequent query for hotels in Gangnam yielded the same hotel options as in Hyderabad.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone needed; provided hotel options in Gangnam as requested.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='07c975bd-635e-49cb-b161-3e9a5d19837a'), HumanMessage(c

In [46]:
config = {"configurable": {"thread_id": "test-1"}}
response = agent.invoke({"messages": [HumanMessage(content="find hotels in seoul")]}, config=config)
response['messages'][-1].content

'Here are some hotel options in Seoul:\n\n1. **Grand Hotel** - 5 stars, ₹10,000/night, includes a spa, pool, and gym.\n2. **City Inn** - 4 stars, ₹700/night, features a business center.\n3. **Budget Stay** - 3 stars, ₹200/night, offers free WiFi.\n\nIf you need more details or assistance, just let me know!'

In [49]:
# Based on fraction

def search_hotels(Cityv: str) -> str:
    """search hotels - return long response to use more tokens"""
    return f"""Hotels in {city}
    1. Grand Hotel - 5 star, ₹1000O/night, spa, pool, gym
    2.  City Inn - 4 star, ₹700/night, business center
    3.. Budget Stay - 3 star, ₹200/night, free wifi
    """

# LOW fraction for testing

agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model = ("gpt-4o-mini"),
        trigger = ("fraction",0.005), # 0.5% = ~640 tokens
        keep = ("fraction", 0.002) # 0.2% = ~256 tokens
        )]
            
)

config = {"configurable": {"thread_id": "test-1"}}

# Token Counter 
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# test 

cities = ["hyderabad", "seoul", "gangnam", "mumbai", "pune", "degu", "busan", "kolkota"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config=config,
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens, tokens({fraction:.4%}),{len(response['messages'])} messages")
    print(response["messages"])

hyderabad: ~149 tokens, tokens(0.1164%),4 messages
[HumanMessage(content='find hotels in hyderabad', additional_kwargs={}, response_metadata={}, id='f0b8f2c5-8401-4fdb-b29b-4b8731571935'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 55, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b1a1d774fb', 'id': 'chatcmpl-Dmd0O9yivBLSenZIpSu8WARihtCzd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8d00-0dc5-75f2-8f6e-68c26f468ceb-0', tool_calls=[{'name': 'search_hotels', 'args': {'Cityv': 'Hyderabad'}, 'id': 'call_83uMWbwPug4k7rxCYGFJ1j0O', 'type': 'tool_call'}], invalid_tool_calls=[], usage

# Human-in-the-loop Middleware 

Pause agent execution for human approval, editing or rejection of a toolcalls before they execute. Human-in-the-loop is useful for the following:
- high-stakes operations requiring human approval
- compliance workflows where human oversight is mandatory 
- long-running conversations where human feedback guides the agent

In [50]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [52]:
agent = create_agent(
    model = "gpt-4o",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ] 
)

In [53]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step - 1 : Request
result = agent.invoke(
    {"messages":[HumanMessage(content="send email to varsha@test.com with subject 'heyyo' and body 'hii'")]},
    config = config
)

In [54]:
result

{'messages': [HumanMessage(content="send email to varsha@test.com with subject 'heyyo' and body 'hii'", additional_kwargs={}, response_metadata={}, id='da36d766-9880-45aa-b53e-24d412c3bfde'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 98, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2c1bd5cc0e', 'id': 'chatcmpl-Dmj6PUNJLPOkorY3d2Sme3QhDwGPV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8e65-a90d-7e61-a018-82eb05b7caed-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'varsha@test.com', 'subject': 'heyyo', 'body': 'hii'}, 'id': 'call_IzyQ1MDAS8hA25KmwdPDEwM3', 'ty

In [56]:
# Step - 2: Approve

from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving..")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config = config
    )
    print(f"Result : {result['messages'][-1].content}")

Paused! Approving..
Result : The email has been sent to varsha@test.com with the subject 'heyyo' and body 'hii'.
